# Proposed Model 3 — Thermal Target Re-acquisition Lab\n\n**Pipeline:** RGB video → synthetic thermal → target selection → SAM2 tracking → lock-loss detection → selectable re-acquisition → evaluation.\n\nChoose Sunny / Cloudy / Rainy / Night, a re-acquisition algorithm, and TherA steps directly in the app.\n

In [ ]:
import os, sys, subprocess, urllib.request, shutil
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Colab: Runtime → Change runtime type → T4 GPU.")

REPO = Path("/content/Target_Locker")

# Target_Locker is now the only GitHub repository needed.
if not REPO.exists():
    subprocess.run([
        "git","clone","--depth","1","--branch","main",
        "https://github.com/CptImtiaz/Target_Locker.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","origin","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","main"], check=True)
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only","origin","main"], check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q","-U",
    "-r",str(REPO/"proposed_model_3"/"requirements.txt")
], check=True)

from huggingface_hub import snapshot_download

# Large neural-network weights cannot be committed as ordinary Git files.
# Download only the TherA inference assets used by Proposed Model 2.
THERA_WEIGHTS = Path("/content/thera_weights")
THERA_WEIGHTS.mkdir(parents=True, exist_ok=True)

print("⬇️ Checking/downloading ThermalConversion assets from Imtiaz807/ThermalConversion...")
snapshot_download(
    repo_id="Imtiaz807/ThermalConversion",
    local_dir=str(THERA_WEIGHTS),
    allow_patterns=[
        "model.pt",
        "checkpoint/*",
        "checkpoint/**",
        "merged_models/*",
        "merged_models/**",
        "stable-diffusion/*",
        "stable-diffusion/**",
        "palettes/*",
        "palettes/**",
        "*SUNNY.pt","**/SUNNY.pt",
        "*CLOUDY.pt","**/CLOUDY.pt",
        "*RAINY.pt","**/RAINY.pt",
        "*NIGHT.pt","**/NIGHT.pt",
    ],
)

# Normalize checkpoint location because TherA distributions have used
# both weights/model.pt and weights/checkpoint/model.pt layouts.
checkpoint_dir = THERA_WEIGHTS / "checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_model = checkpoint_dir / "model.pt"
root_model = THERA_WEIGHTS / "model.pt"

if not checkpoint_model.exists() and root_model.exists():
    shutil.copy2(root_model, checkpoint_model)

if not checkpoint_model.exists():
    candidates = [p for p in THERA_WEIGHTS.rglob("model.pt") if p.is_file()]
    if candidates:
        shutil.copy2(candidates[0], checkpoint_model)

cache_dir = THERA_WEIGHTS / "palettes"
cache_dir.mkdir(parents=True, exist_ok=True)

# Normalize all four documented TherA reference conditions.
for condition in ["SUNNY", "CLOUDY", "RAINY", "NIGHT"]:
    dst = cache_dir / f"{condition}.pt"
    if not dst.exists():
        matches = [
            p for p in THERA_WEIGHTS.rglob(f"{condition}.pt")
            if p.is_file() and p.resolve() != dst.resolve()
        ]
        if matches:
            shutil.copy2(matches[0], dst)

required = [
    checkpoint_model,
    THERA_WEIGHTS / "merged_models" / "unet",
    THERA_WEIGHTS / "merged_models" / "adapter",
    THERA_WEIGHTS / "stable-diffusion" / "vae",
    THERA_WEIGHTS / "stable-diffusion" / "scheduler",
]

missing = [str(p) for p in required if not p.exists()]
available_conditions = [
    c for c in ["SUNNY","CLOUDY","RAINY","NIGHT"]
    if (cache_dir / f"{c}.pt").exists()
]

if missing:
    raise RuntimeError(
        "TherA model download is incomplete. Missing:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not available_conditions:
    raise RuntimeError(
        "No TherA reference cache was found. Expected palettes/SUNNY.pt, CLOUDY.pt, RAINY.pt, and NIGHT.pt files."
    )

SAM_CKPT = Path("/content/sam2.1_hiera_tiny.pt")
if not SAM_CKPT.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        SAM_CKPT
    )

sys.path.insert(0, str(REPO/"proposed_model_2"))

print("✅ Proposed Model 3 ready")
print("Bundled TherA runtime:", REPO/"proposed_model_3"/"thera_runtime.py")
print("TherA weights:", THERA_WEIGHTS)
print("Available conditions:", ", ".join(available_conditions))
print("SAM2 checkpoint:", SAM_CKPT)







In [ ]:
import os, sys\nfrom pathlib import Path\n\nREPO = Path("/content/Target_Locker")\nsys.path.insert(0, str(REPO/"proposed_model_3"))\nos.chdir(REPO/"proposed_model_3")\n\nexec((REPO/"proposed_model_3"/"app_interface.py").read_text(), globals())\n